# Dataset Grid: Hierarchical Click Curves by Image Position

This notebook plots **hierarchically bootstrapped Dice-vs-click curves** for selected image positions
(default indices `0,1,2,4,9`) across datasets in a procedure.

- Each subplot = one dataset
- Each line = one selected image index
- Shaded band = 95% CI from hierarchical bootstrap (subset resampling within task, then across tasks)


In [ ]:
from __future__ import annotations

import math
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import MaxNLocator, MultipleLocator

from experiments.analysis.hierarchical_ci import hierarchical_bootstrap_curve_2d
from experiments.analysis.planb_click_curves import build_task_curves_by_image
from experiments.analysis.planb_utils import iter_planb_subset_dirs


In [ ]:
# -----------------------------
# Config
# -----------------------------
REPO_ROOT = Path('/data/ddmg/mvseg-ordering')

PROCEDURE = 'random_vs_uncertainty_final'
ABLATION = 'pretrained_baseline5p'
POLICY_NAME = 'random'

IMAGE_INDICES = [0, 1, 2, 4, 9]
LEGEND_ONE_INDEXED = True

N_BOOT = 1000
SEED = 23

N_COLS = 3
Y_LIM = (0.40, 0.90)

SAVE_FIGURE = False
OUT_DIR = REPO_ROOT / 'figures' / 'click_curve_dataset_grid'
OUT_NAME = f"{PROCEDURE.replace('/', '_')}_{ABLATION}_{POLICY_NAME}_dataset_grid.png"


In [ ]:
def legend_label_from_index(img_index: int, one_indexed: bool = True) -> str:
    # Paper-facing legend text.
    return f"Image {img_index + 1}" if one_indexed else f"Image {img_index}"


def collect_subset_entries(*, repo_root: Path, procedure: str, ablation: str, policy_name: str) -> list[dict[str, object]]:
    entries = list(
        iter_planb_subset_dirs(
            repo_root=repo_root,
            procedure=procedure,
            ablation=ablation,
            policy=policy_name,
            include_families=None,
        )
    )
    if not entries:
        raise FileNotFoundError(
            f"No Plan B subset dirs found for procedure={procedure}, ablation={ablation}, policy={policy_name}"
        )
    return entries


def bootstrap_family_curves(
    family_entries: list[dict[str, object]],
    image_indices: list[int],
    *,
    n_boot: int,
    seed: int,
) -> dict[int, dict[str, pd.Series]]:
    per_image_tables = build_task_curves_by_image(family_entries, image_indices)
    out: dict[int, dict[str, pd.Series]] = {}
    for img in image_indices:
        task_tables = per_image_tables.get(int(img), {})
        if not task_tables:
            continue
        boot = hierarchical_bootstrap_curve_2d(
            task_tables,
            n_boot=n_boot,
            seed=seed,
        )
        out[int(img)] = {
            'mean': boot['mean'],
            'ci_lo': boot['ci_lo'],
            'ci_hi': boot['ci_hi'],
        }
    return out


In [ ]:
entries = collect_subset_entries(
    repo_root=REPO_ROOT,
    procedure=PROCEDURE,
    ablation=ABLATION,
    policy_name=POLICY_NAME,
)

families_with_data = sorted({str(m['family']) for m in entries})
print('Families found:', families_with_data)
print('Total subset dirs:', len(entries))

family_to_entries: dict[str, list[dict[str, object]]] = {fam: [] for fam in families_with_data}
for meta in entries:
    family_to_entries[str(meta['family'])].append(meta)

family_boot: dict[str, dict[int, dict[str, pd.Series]]] = {}
for fam in families_with_data:
    fam_entries = family_to_entries[fam]
    family_boot[fam] = bootstrap_family_curves(
        fam_entries,
        IMAGE_INDICES,
        n_boot=N_BOOT,
        seed=SEED,
    )
    print(f"[done] {fam}: subset_dirs={len(fam_entries)} image_curves={sorted(family_boot[fam].keys())}")


In [ ]:
families = [f for f in families_with_data if family_boot.get(f)]
if not families:
    raise RuntimeError('No families had bootstrapped curves to plot.')

n_cols = int(max(1, N_COLS))
n_rows = int(math.ceil(len(families) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(7.2 * n_cols, 4.8 * n_rows), squeeze=False)
axes_flat = axes.ravel()
cmap = plt.get_cmap('tab10')

for ax_idx, family in enumerate(families):
    ax = axes_flat[ax_idx]
    boot_by_image = family_boot[family]

    for color_idx, img in enumerate(IMAGE_INDICES):
        if int(img) not in boot_by_image:
            continue
        curve = boot_by_image[int(img)]
        mean_curve = curve['mean']
        lo_curve = curve['ci_lo']
        hi_curve = curve['ci_hi']
        color = cmap(color_idx % 10)
        label = legend_label_from_index(int(img), one_indexed=LEGEND_ONE_INDEXED)

        ax.plot(mean_curve.index, mean_curve.values, linewidth=2.2, color=color, label=label)
        ax.fill_between(mean_curve.index, lo_curve.values, hi_curve.values, alpha=0.18, color=color)

    ax.set_title(f"{family}", fontsize=12)
    ax.set_xlabel('Click Iteration', fontsize=11)
    ax.set_ylabel('Dice Score', fontsize=11)
    ax.grid(True, linestyle='--', linewidth=0.6, alpha=0.35)
    ax.xaxis.set_major_locator(MultipleLocator(1))
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if Y_LIM is not None:
        ax.set_ylim(*Y_LIM)

    # Mark initial (no-click) position if present.
    if any((-1 in c['mean'].index) for c in boot_by_image.values()):
        ax.axvline(-1, color='#888888', linestyle='--', linewidth=1, alpha=0.55)

# Hide empty panels.
for j in range(len(families), len(axes_flat)):
    axes_flat[j].axis('off')

# Single legend for the whole figure (paper cleaner).
handles, labels = axes_flat[0].get_legend_handles_labels()
if handles:
    fig.legend(handles, labels, loc='upper center', ncol=min(len(labels), 5), frameon=False, fontsize=11)

paper_title = (
    'Hierarchical Bootstrap Click Curves Across Datasets '
    f'({POLICY_NAME}, selected image positions)'
)
fig.suptitle(paper_title, fontsize=15, y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.965))

if SAVE_FIGURE:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    out_path = OUT_DIR / OUT_NAME
    fig.savefig(out_path, dpi=220, bbox_inches='tight')
    print(f"Saved: {out_path}")

plt.show()
